In [3]:
import cobra
from corda import CORDA

import json

import pandas as pd
import random


import sys
sys.path.insert(1, '/home/hratch/Projects/human_me/')
from human_me.core import biomass

ImportError: cannot import name 'correct_m_model'

In [4]:
build_files_path = '/data2/hratch/human_me/build/'
full_model = cobra.io.read_sbml_model('/data2/hratch/human_me/prebuild/recon2_2.xml')
full_model = biomass.correct_m_biomass(full_model) # correct the biomass objective
biomass_reactions = [r.id for r in full_model.reactions if 'biomass' in r.id]


# vN = full_model.optimize()
# res = pd.DataFrame(index = v1.fluxes.index, data = {'v1': v1.fluxes.tolist(), 'v2': v2.fluxes.tolist()})
# res['diff'] = res['v1'] - res['v2'].abs().tolist()
# res.loc[biomass_reactions,]

In [5]:
rmd = json.load(open(build_files_path + "required_metabolic_model_metabolites.json"))
rmd = [v for k,v in rmd.items() if k == 'c']
rmd = [item for sublist in rmd for item in sublist]

In [6]:
def flatten_list(t):
    #https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-list-of-lists
    return [item for sublist in t for item in sublist]

# Build Toy Models

# Build a Random Toy Model

In [31]:
def build_random_toy_model(full_model, rmd, max_score = 3, frac_reactions = 0.01):
    '''Generate small toy model from full Recon2.2'''

    # generate reaction confidence scores
    n_reactions = len(full_model.reactions)
    n_reactions_to_keep = round(frac_reactions*n_reactions)

    one_reaction = [r.id for r in full_model.reactions if len(r.genes)==1]
    reactions_to_exclude = random.sample(one_reaction, round(len(one_reaction)*.25)) # decreases prob of choosing a reaction with a gene by 4x
    reactions_to_exclude += [r.id for r in full_model.reactions if len(r.genes)>1]


    population = sorted(set([r.id for r in full_model.reactions]).difference(reactions_to_exclude))
    reactions_to_keep = random.sample(population, k = n_reactions_to_keep)

    conf = {}
    for r in full_model.reactions: 
        if r.id not in reactions_to_keep:
            conf[r.id] = -1
        else:
            conf[r.id] = random.choice(list(range(max_score +1)))
    for r_id in biomass_reactions:
        conf[r_id] = 3
    
    print('Extract model')
    opt = CORDA(full_model, conf, met_prod = rmd)
    opt.build()
    print(opt)
    
    print('Generate cobra model')
    toy_model = opt.cobra_model('toy_model')
    
    return toy_model

In [33]:
iter_, max_iter = 0, 10
first = True
min_growth = 1e-3
toy_model = cobra.Model('')
opt_val = toy_model.slim_optimize()

while (iter_ < max_iter) and opt_val < min_growth:
    print('iteration: {}'.format(iter_))
    
    toy_model = build_random_toy_model(full_model, rmd)
    opt_val = toy_model.slim_optimize()

    print('Growth value: {}'.format(opt_val))
    print('--------')
    iter_ += 1

# print(opt_val)
# if opt_val >= min_growth:
#     cobra.io.write_sbml_model(cobra_model = toy_model, 
#                               filename = '/data2/hratch/human_me/input_files/toy_model.xml')

In [ ]:
minimal_model = build_toy_model(full_model, rmd, max_score = 3, frac_reactions = 0)

# Generate a core model

A model that includes nucleotide and amino acid synthesis, and central carbon metabolism

In [31]:
mem = False # minimal essential media, or all exchange reactions

In [32]:
core_groups = ['Alanine and aspartate metabolism', 'Lysine metabolism', 
               'Glycine, serine, alanine and threonine metabolism',
              'Methionine and cysteine metabolism', 'Tyrosine metabolism', 'Histidine metabolism', 
               'Cysteine Metabolism', 'Valine, leucine, and isoleucine metabolism', 'Tryptophan metabolism', 
               'Arginine and Proline Metabolism', 'Glutamate metabolism', 'Glutathione metabolism', 
               'Phenylalanine metabolism', 
          # amino acids^
          # nucleotides
          #'Nucleotide interconversion', 'Nucleotide salvage pathway',
          'Pyrimidine catabolism', 'Pyrimidine synthesis', 'Purine catabolism', 'Purine synthesis',
          # transport
#           'Transport, peroxisomal', 'Transport, extracellular', 
#           'Transport, lysosomal', 'Transport, endoplasmic reticular', 'Transport, nuclear', 
#           'Transport, golgi apparatus', 'Transport, mitochondrial',
#                'Exchange/demand reaction',
          # central carbon
         'Glycolysis/gluconeogenesis', 'Pentose phosphate pathway', 'Citric acid cycle', 
               'Oxidative phosphorylation']
 
reactions = list(set([r.id for r in flatten_list([g.members for g in full_model.groups if g.id in core_groups])]))

# MEM minimal essential medium: https://www.thermofisher.com/us/en/home/technical-resources/media-formulation.204.html
if mem:
    reactions += ['EX_arg_L_b', 'EX_arg_L_LPAREN_e_RPAREN_', 'EX_his_L_b', 'EX_his_L_LPAREN_e_RPAREN_', 
                 'EX_cys_L_b', 'EX_cys_L_LPAREN_e_RPAREN_', 'EX_gln_L_LPAREN_e_RPAREN_', 'EX_gln_L_b', 
                 'EX_leu_L_b', 'EX_leu_L_LPAREN_e_RPAREN_', 'EX_tyr_L_LPAREN_e_RPAREN_', 'EX_tyr_L_b',
                   'EX_ile_L_LPAREN_e_RPAREN_', 'EX_ile_L_b', 'EX_lys_L_LPAREN_e_RPAREN_', 'EX_lys_L_b', 
                   'EX_met_L_LPAREN_e_RPAREN_', 'EX_met_L_b', 'EX_phe_L_b', 'EX_phe_L_LPAREN_e_RPAREN_', 
                   'EX_thr_L_b', 'EX_thr_L_LPAREN_e_RPAREN_', 'EX_trp_L_b', 'EX_trp_L_LPAREN_e_RPAREN_', 
                   'EX_val_L_LPAREN_e_RPAREN_', 'EX_val_L_b'] # essential AA's
    reactions += ['EX_glc_LPAREN_e_RPAREN_', 'EX_glc_D_b', 'EX_h2o_LPAREN_e_RPAREN_', 'EX_h2o_b', 
                'EX_cl_b', 'EX_cl_LPAREN_e_RPAREN_', 'EX_pnto_R_LPAREN_e_RPAREN_', 'EX_pnto_R_b', 'DM_pnto_R', 
                'EX_fol_LPAREN_e_RPAREN_', 'EX_fol_b',  'EX_pydx_LPAREN_e_RPAREN_', 'EX_pydx_b', 
                 'EX_ribflv_b', 'EX_ribflv_LPAREN_e_RPAREN_', 'EX_thm_LPAREN_e_RPAREN_', 'EX_thm_b', 
                 'EX_inost_LPAREN_e_RPAREN_', 'EX_inost_b', 'EX_ca2_LPAREN_e_RPAREN_', 'EX_ca2_b', 
                 'EX_k_b', 'EX_k_LPAREN_e_RPAREN_', 'EX_so4_LPAREN_e_RPAREN_', 'EX_so4_b', 
                  'EX_na1_LPAREN_e_RPAREN_', 'EX_na1_b', 'EX_pi_b', 'EX_pi_LPAREN_e_RPAREN_']
else:
    reactions += [r.id for r in full_model.reactions if r.id.startswith('EX_')]
    

reactions_to_exclude = [r.id for r in full_model.reactions if r.id not in reactions]

In [33]:
# generate confidence scores
conf = {}
for r in full_model.reactions: 
    if r.id not in reactions:
        conf[r.id] = -1
    else:
        conf[r.id] = 3
for r_id in biomass_reactions:
    conf[r_id] = 3

In [35]:
print('Extract model')
opt = CORDA(full_model, conf, met_prod = rmd)
opt.build()
print(opt)

print('Generate cobra model')
core_model = opt.cobra_model('core_model')

In [ ]:
core_model.slim_optimize()

In [ ]:
fn = '/data2/hratch/human_me/other/core' 
if mem:
    fn += '_mem'

cobra.io.write_sbml_model(cobra_model = core_model, filename = fn + #'_v2.xml')

Generate a central carbon metabolism ONLY model

In [7]:
mem = False # minimal essential media, or all exchange reactions

In [8]:
core_groups = ['Glycolysis/gluconeogenesis', 'Pentose phosphate pathway', 'Citric acid cycle', 'Oxidative phosphorylation']
 
reactions = list(set([r.id for r in flatten_list([g.members for g in full_model.groups if g.id in core_groups])]))
reactions.append('biomass_reaction')

# MEM minimal essential medium: https://www.thermofisher.com/us/en/home/technical-resources/media-formulation.204.html
if mem:
    reactions += ['EX_arg_L_b', 'EX_arg_L_LPAREN_e_RPAREN_', 'EX_his_L_b', 'EX_his_L_LPAREN_e_RPAREN_', 
                 'EX_cys_L_b', 'EX_cys_L_LPAREN_e_RPAREN_', 'EX_gln_L_LPAREN_e_RPAREN_', 'EX_gln_L_b', 
                 'EX_leu_L_b', 'EX_leu_L_LPAREN_e_RPAREN_', 'EX_tyr_L_LPAREN_e_RPAREN_', 'EX_tyr_L_b',
                   'EX_ile_L_LPAREN_e_RPAREN_', 'EX_ile_L_b', 'EX_lys_L_LPAREN_e_RPAREN_', 'EX_lys_L_b', 
                   'EX_met_L_LPAREN_e_RPAREN_', 'EX_met_L_b', 'EX_phe_L_b', 'EX_phe_L_LPAREN_e_RPAREN_', 
                   'EX_thr_L_b', 'EX_thr_L_LPAREN_e_RPAREN_', 'EX_trp_L_b', 'EX_trp_L_LPAREN_e_RPAREN_', 
                   'EX_val_L_LPAREN_e_RPAREN_', 'EX_val_L_b'] # essential AA's
    reactions += ['EX_glc_LPAREN_e_RPAREN_', 'EX_glc_D_b', 'EX_h2o_LPAREN_e_RPAREN_', 'EX_h2o_b', 
                'EX_cl_b', 'EX_cl_LPAREN_e_RPAREN_', 'EX_pnto_R_LPAREN_e_RPAREN_', 'EX_pnto_R_b', 'DM_pnto_R', 
                'EX_fol_LPAREN_e_RPAREN_', 'EX_fol_b',  'EX_pydx_LPAREN_e_RPAREN_', 'EX_pydx_b', 
                 'EX_ribflv_b', 'EX_ribflv_LPAREN_e_RPAREN_', 'EX_thm_LPAREN_e_RPAREN_', 'EX_thm_b', 
                 'EX_inost_LPAREN_e_RPAREN_', 'EX_inost_b', 'EX_ca2_LPAREN_e_RPAREN_', 'EX_ca2_b', 
                 'EX_k_b', 'EX_k_LPAREN_e_RPAREN_', 'EX_so4_LPAREN_e_RPAREN_', 'EX_so4_b', 
                  'EX_na1_LPAREN_e_RPAREN_', 'EX_na1_b', 'EX_pi_b', 'EX_pi_LPAREN_e_RPAREN_']
else:
    reactions += [r.id for r in full_model.reactions if r.id.startswith('EX_')]

reactions_to_exclude = [r.id for r in full_model.reactions if r.id not in reactions]

In [9]:
# generate confidence scores
conf = {}
for r in full_model.reactions: 
    if r.id not in reactions:
        conf[r.id] = -1
    else:
        conf[r.id] = 3
for r_id in biomass_reactions:
    conf[r_id] = 3

In [10]:
print('Extract model')
opt = CORDA(full_model, conf, met_prod = rmd)
opt.build()
print(opt)

print('Generate cobra model')
central_model = opt.cobra_model('central_carbon_model')

Extract model
build status: reconstruction complete
Inc. reactions: 1963/8555
 - unclear: 0/0
 - exclude: 391/6981
 - low and medium: 0/0
 - high: 1572/1574

Generate cobra model


cobra/core/group.py:110 UserWarning: need to pass in a list


In [11]:
central_model.slim_optimize()

444.5191511198606

In [12]:
len(central_model.reactions)

1916

In [13]:
sln = central_model.optimize()
sln.fluxes.loc[biomass_reactions,]

EX_biomass_c            444.519151
biomass_reaction        444.519151
biomass_protein         313.830521
biomass_DNA               6.223268
biomass_RNA              25.782111
biomass_carbohydrate     31.560860
biomass_lipid            43.118358
biomass_other            24.004034
Name: fluxes, dtype: float64

In [25]:
fn = '/data2/hratch/human_me/other/central' 
if mem:
    fn += '_mem'

# cobra.io.write_sbml_model(cobra_model = central_model, filename = fn + '_v2.xml')
human_me.io.write_metabolic_model(m_model = central_model, file_name = fn + '_v2.xml')